# Quick start — Earth Engine via the `EarthLens` facade

This notebook is the script-form quickstart for the GEE backend. The per-category
notebooks in this directory each use `earthlens.gee.GEE(...)` directly; this one
demonstrates the **unified `EarthLens` facade** entry point — the same call
pattern that works for the CHC, S3, and ECMWF backends.

It also shows the two most common follow-on patterns as commented blocks:

- bypassing the facade with `GEE(...)` directly;
- queueing an asynchronous Drive export for an AOI too large for the synchronous
  URL path.

The notebook reads service-account credentials from `GEE_SERVICE_ACCOUNT` and
`GEE_SERVICE_KEY` environment variables. See
[Service account setup](../../reference/gee/service-account-setup.md) for the
one-time provisioning steps.

In [1]:
import os

from earthlens import EarthLens
from earthlens.gee import Catalog

SERVICE_ACCOUNT = os.environ.get(
    'GEE_SERVICE_ACCOUNT', 'my-sa@my-project.iam.gserviceaccount.com'
)
SERVICE_KEY = os.environ.get('GEE_SERVICE_KEY', '/path/to/key.json')

2026-05-18 01:17:56 | INFO | pyramids.base.config | Logging is configured.


## Browse the catalog (no network, no auth)

The catalog ships with the package and parses ~57 k lines of curated YAML in
~3 s cold / ~1 ms warm. Use it to discover what's available and look up a
band's units before issuing a download.

In [2]:
catalog = Catalog()
print(
    f'{len(catalog.datasets)} curated datasets; '
    f'{len(catalog.available_datasets)} in the Earth Engine STAC index'
)

chirps = catalog.get_dataset('UCSB-CHG/CHIRPS/DAILY')
print(f'CHIRPS: ee_type={chirps.ee_type}, cadence={chirps.cadence}')
print(f'  bands: {list(chirps.bands)}')
print(
    f"  precipitation units: {catalog.get_band('UCSB-CHG/CHIRPS/DAILY', 'precipitation').units}"
)

1104 curated datasets; 1104 in the Earth Engine STAC index
CHIRPS: ee_type=image_collection, cadence=interval=1 unit='day'
  bands: ['precipitation']
  precipitation units: mm/d


## Download via the `EarthLens` facade

Monthly CHIRPS precipitation composites over a small bbox in Egypt for
Jun-Aug 2020. `EarthLens(...)` resolves `data_source="gee"` to the GEE
backend via its `_LazyRegistry`; the remaining kwargs forward verbatim to
`GEE(...)`. Output is one GeoTIFF per month (three files total) under
`out/quickstart/`.

In [3]:
el = EarthLens(
    data_source='gee',
    start='2020-06-01',
    end='2020-08-31',
    temporal_resolution='monthly',
    variables={'UCSB-CHG/CHIRPS/DAILY': ['precipitation']},
    lat_lim=[28.0, 32.0],
    lon_lim=[30.0, 34.0],
    path='out/quickstart',
    scale=5566,
    service_account=SERVICE_ACCOUNT,
    service_key=SERVICE_KEY,
)
paths = el.download(progress_bar=False)
for p in paths:
    print(p)

2026-05-18 01:18:04.322 | INFO     | earthlens.gee.backend:_download_one_url_tile:757 - Wrote C:\gdrive\algorithms\remote-sensing\earthlens\docs\examples\google-earth-engine\out\quickstart\UCSB-CHG_CHIRPS_DAILY_precipitation_20200601.tif (19905 bytes)


2026-05-18 01:18:05.726 | INFO     | earthlens.gee.backend:_download_one_url_tile:757 - Wrote C:\gdrive\algorithms\remote-sensing\earthlens\docs\examples\google-earth-engine\out\quickstart\UCSB-CHG_CHIRPS_DAILY_precipitation_20200701.tif (5351 bytes)


2026-05-18 01:18:07.076 | INFO     | earthlens.gee.backend:_download_one_url_tile:757 - Wrote C:\gdrive\algorithms\remote-sensing\earthlens\docs\examples\google-earth-engine\out\quickstart\UCSB-CHG_CHIRPS_DAILY_precipitation_20200801.tif (661 bytes)


C:\gdrive\algorithms\remote-sensing\earthlens\docs\examples\google-earth-engine\out\quickstart\UCSB-CHG_CHIRPS_DAILY_precipitation_20200601.tif
C:\gdrive\algorithms\remote-sensing\earthlens\docs\examples\google-earth-engine\out\quickstart\UCSB-CHG_CHIRPS_DAILY_precipitation_20200701.tif
C:\gdrive\algorithms\remote-sensing\earthlens\docs\examples\google-earth-engine\out\quickstart\UCSB-CHG_CHIRPS_DAILY_precipitation_20200801.tif


## Alternative — bypass the facade with `GEE(...)`

The per-category notebooks in this directory all use this form. It's a thinner
call (no registry lookup) and gives the IDE precise type hints for the
backend's kwargs.

In [4]:
# Equivalent to the cell above:
#
# from earthlens.gee import GEE
# gee = GEE(
#     start='2020-06-01', end='2020-08-31', temporal_resolution='monthly',
#     variables={'UCSB-CHG/CHIRPS/DAILY': ['precipitation']},
#     lat_lim=[28.0, 32.0], lon_lim=[30.0, 34.0],
#     path='out/quickstart', scale=5566,
#     service_account=SERVICE_ACCOUNT, service_key=SERVICE_KEY,
# )
# gee.download(progress_bar=False)

## Alternative — asynchronous Drive export for large AOIs

The default `export_via="url"` uses `getDownloadURL` which is capped at
32768 px per axis. For larger AOIs queue an
`ee.batch.Export.image.toDrive` task instead; `download()` will block while
it runs and surface the destination string when it's done.

For non-blocking submission see `track-batch-exports.ipynb` and the
per-category notebooks' "Tracking submitted jobs" sections — pass
`wait_for_export=False` and call `wait_for_task_id(...)` later.

In [5]:
# el = EarthLens(
#     data_source='gee',
#     start='2023-01-01', end='2023-12-31', temporal_resolution='monthly',
#     variables={'COPERNICUS/S2_SR_HARMONIZED': ['B4', 'B8']},
#     lat_lim=[51.0, 53.0], lon_lim=[4.0, 7.0],
#     scale=10, export_via='drive', drive_folder='ee_exports',
#     service_account=SERVICE_ACCOUNT, service_key=SERVICE_KEY,
# )
# locations = el.download()